In [19]:
import os
from dotenv import load_dotenv
import requests
import base64 
import json
from urllib.parse import quote
import time
import pandas as pd

load_dotenv()

client_id = os.getenv("CLIENT_ID")
client_secret = os.getenv("CLIENT_SECRET")
base_url = "https://api.spotify.com/v1"
redirect_uri = os.getenv("REDIRECT_URI")
scope = "playlist-read-public"
token_url = "https://accounts.spotify.com/api/token"

class SpotifyAPI:
    def __init__(self) -> object:
        self.client_id = client_id
        self.client_secret = client_secret
        self.base_url = base_url
        self.token_url = token_url
        
    def get_access_token(self) -> str:
        response = requests.post(
            self.token_url,
            data={"grant_type": "client_credentials"},
            auth=(self.client_id, self.client_secret)
        )
        return response.json()["access_token"]

    def make_request(self, endpoint, params=None, body=None):
        
        url = f"{self.base_url}/{endpoint}"
        if params:
            for param in params:
                url+=f"?{param}"
        access_token = self.get_access_token()
        res = requests.get(
            url,
            headers={
                "Authorization" : f"Bearer {access_token}"
            }
        )
        print(url)
        if res.status_code == 200:
            return res.json()
        else:
            print(f"Failed to fetch: {res.status_code}")


    def get_categories(self, country="US", limit=2):
        return self.make_request(f"browse/categories?country={country}&limit={limit}")

    def get_playlists(self, category_id, country="US", limit=5):
        return self.make_request(f"search?q=category:{category_id}&type=playlist&limit={limit}")

    def search_song(self, song, artist):
        query = f"track:{song} artist:{artist}"
        endpoint = f"search?q={quote(query)}&type=track&limit=1"
        res = self.make_request(endpoint=endpoint)
        if not res:
            return None

        tracks = res.get("tracks", [])
        if not tracks:
            return None
 
        if tracks['total'] == 0:
            return -1
        id = self.make_request(endpoint=endpoint)['tracks']['items'][0]
        # print(id)
        return id
    def get_song(self, id):
        return self.make_request(f"tracks/{id}")
    def search_artist(self, artist):
        query = f"artist:{artist}"
        endpoint = f"search?q={quote(query)}&type=artist&limit=1"
        id = self.make_request(endpoint=endpoint)
        
        # return self.make_request(f"tracks/{id}")
        return id   
    def get_artist(self, id):
        return self.make_request(f"artists/{id}")

In [6]:
base_url_last_fm="https://ws.audioscrobbler.com/2.0"
last_fm_api_key = os.getenv("LAST_FM_API_KEY")
last_fm_shared_secret = os.getenv("LAST_FM_SHARED_SECRET")
mb_base = f"https://musicbrainz.org/ws/2"
import requests
from requests.adapters import HTTPAdapter, Retry

session = requests.Session()

retries = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    raise_on_status=False,
)

session.mount("https://", HTTPAdapter(max_retries=retries))

import musicbrainzngs

class LAST_FM_API():
    def __init__(self) -> object:
        self.base_url_last_fm = base_url_last_fm
        self.last_fm_api_key = last_fm_api_key
        self.last_fm_shared_secret = last_fm_shared_secret
        self.mb_base = mb_base
        self.session = requests.Session()

        
    def make_request(self, endpoint):
        url = f"{base_url_last_fm}/{endpoint}&api_key={last_fm_api_key}&format=json"
        res = requests.get(url)
        print(url)
        return res.json()
    
    def mb_make_request(self, endpoint, retries=3):
        url = f"{self.mb_base}/{endpoint}&fmt=json"
        print(url)

        for attempt in range(1, retries + 1):
            try:
                res = self.session.get(url, timeout=20)

                # Handle rate limiting
                if res.status_code == 503:
                    wait = 1 * attempt
                    print(f"Rate limit hit. Retrying in {wait}s...")
                    time.sleep(wait)
                    continue

                if res.status_code != 200:
                    print(f"HTTP {res.status_code}: {res.reason}")
                    return None

                try:
                    return res.json()
                except ValueError:
                    print("Error decoding JSON")
                    return None

            except (requests.exceptions.Timeout,
                    requests.exceptions.ConnectionError) as e:
                print(f"Network error: {e}. Attempt {attempt}/{retries}")
                time.sleep(1)
        
        print("Max retries exceeded.")
        return None
    

    def get_artist_info(self, artist):
        # query = f"artist={artist}"
        endpoint = f"?method=artist.getinfo&artist={quote(artist)}"
        print(endpoint)
        return self.make_request(endpoint)
    

    def get_mbid(self, link):
        endpoint = f"url?resource={link}&inc=artist-rels"
        res = self.mb_make_request(endpoint)

        if not res:
            return None

        relations = res.get("relations", [])
        if not relations:
            return None

        # Safely navigate nested objects
        artist = relations[0].get("artist", {})
        return artist.get("id")
    
    
    def mb_get_artist_tag(self, id):
        endpoint = f"artist/{id}?inc=tags"
        tags = self.mb_make_request(endpoint)['tags']
        res = self.mb_make_request(endpoint)
        if not res:
            return None
        tags = res.get('tags', [])
        if not tags:
            return None
        top = max(tags, key=lambda tag: tag.get('count')).get('name', "")
        return top
# last = LAST_FM_API()
# # last.get_artist_info("red hot chili peppers")
# carey = "https://open.spotify.com/artist/4iHNK0tOyZPYnBU7nGAgpQ"

# id = last.get_mbid(carey)
# print(id)
# last.mb_get_artist(id)

In [4]:
import billboard
chart = billboard.ChartData('hot-100')

In [8]:
# need to work on cleaning up artist names properly to send into spotify api, 
# as of now If it isnt a match to the spotify search, we just skip the song
sp = SpotifyAPI()
last = LAST_FM_API()
song_ids = []
artists = []
for song in chart:
    print(song)
    artist = song.artist
    if "Featuring" in artist:
        artist = artist.split("Featuring")[0]
    if artist == "JEONGYEON, JIHYO & CHAEYOUNG Of TWICE":
        artist = "TWICE"
    res = sp.search_song(song.title, artist)
    #if the song is found, add song and artist
    if res and res!= -1:
        song_ids.append(res['id'])
        existing_ids = {a["id"] for a in artists}

        for item in res.get("artists", []):
            artist_id = item.get("id")
            artist_url = item.get("external_urls", {}).get("spotify")
            artist_name = item.get("name")
            if artist_id and artist_id not in existing_ids:
                artists.append({
                    "name": artist_name,
                    "id": artist_id,
                    "url": artist_url
                })
                existing_ids.add(artist_id)

# songs
# sp=SpotifyAPI()
# sp.search_song(chart[0].title, chart[0].artist)


'The Fate Of Ophelia' by Taylor Swift
'Golden' by HUNTR/X: EJAE, Audrey Nuna & REI AMI
'Ordinary' by Alex Warren
'Man I Need' by Olivia Dean
'Opalite' by Taylor Swift
'Daisies' by Justin Bieber
'Mutt' by Leon Thomas
'Folded' by Kehlani
'I Got Better' by Morgan Wallen
'Back To Friends' by sombr
'All I Want For Christmas Is You' by Mariah Carey
'Travelin' Soldier' by Cody Johnson & The Rockin' CJB
'Last Christmas' by Wham!
'What I Want' by Morgan Wallen Featuring Tate McRae
'Soda Pop' by Saja Boys: Andrew Choi, Neckwav, Danny Chung, Kevin Woo & samUIL Lee
'It Depends' by Chris Brown Featuring Bryson Tiller
'Elizabeth Taylor' by Taylor Swift
'Tears' by Sabrina Carpenter
'Rockin' Around The Christmas Tree' by Brenda Lee
'Jingle Bell Rock' by Bobby Helms
'Choosin' Texas' by Ella Langley
'Love Me Not' by Ravyn Lenae
'Manchild' by Sabrina Carpenter
'Your Idol' by Saja Boys: Andrew Choi, Neckwav, Danny Chung, Kevin Woo & samUIL Lee
'Tit For Tat' by Tate McRae
'Wi$h Li$t' by Taylor Swift
'How I

In [6]:
# sp=SpotifyAPI()
# sp.search_artist("Taylor Swift")
# sp.get_artist("06HL4z0CvFAxyc27GXpf02")
# print(artist_urls)
last = LAST_FM_API()
mbids = []
for artist in artists:
    url = artist['url']
#     # print(url)
    mbid = last.get_mbid(url)
    if mbid:
        artist['mbid'] = mbid
        tag = last.mb_get_artist_tag(mbid)
        if tag:
            artist['tag'] = tag
    else:
        print(f"unable to get mbid for {url}")
    time.sleep(1)
    # print(id)
    # last.mb_get_artist(url)

https://musicbrainz.org/ws/2/url?resource=https://open.spotify.com/artist/06HL4z0CvFAxyc27GXpf02&inc=artist-rels&fmt=json
https://musicbrainz.org/ws/2/artist/20244d07-534f-4eff-b4d4-930878889970?inc=tags&fmt=json
https://musicbrainz.org/ws/2/artist/20244d07-534f-4eff-b4d4-930878889970?inc=tags&fmt=json
https://musicbrainz.org/ws/2/url?resource=https://open.spotify.com/artist/2yNNYQBChuox9A5Ka93BIn&inc=artist-rels&fmt=json
https://musicbrainz.org/ws/2/artist/dc7fd424-6ba4-45cc-b407-7716c7bb3605?inc=tags&fmt=json
https://musicbrainz.org/ws/2/artist/dc7fd424-6ba4-45cc-b407-7716c7bb3605?inc=tags&fmt=json
https://musicbrainz.org/ws/2/url?resource=https://open.spotify.com/artist/0RMJOzHDhAKY1o2j0W0vxY&inc=artist-rels&fmt=json
https://musicbrainz.org/ws/2/artist/0f0a0c02-0aa6-4fc0-aad0-bb2e9d5bcbbb?inc=tags&fmt=json
https://musicbrainz.org/ws/2/artist/0f0a0c02-0aa6-4fc0-aad0-bb2e9d5bcbbb?inc=tags&fmt=json
https://musicbrainz.org/ws/2/url?resource=https://open.spotify.com/artist/0Wwji82sLA0Hcv

In [ ]:
base_recco="https://api.reccobeats.com/v1"
headers = {
  'Accept': 'application/json'
}


class ReccoBeats:
  def get_recco_song_details(ids):
    ids_string=','.join(ids)
    res = requests.get(f"{base_recco}/track?ids={ids_string}", headers=headers)
    if(res.status_code == 200):
      return res.json()
    else:
      print(f"request failed with code {res.status_code} due to {res.reason}")

  def get_recco_audio_analysis(id):
    res = requests.get(f"{base_recco}/track/{id}/audio-features", headers=headers)
    if(res.status_code == 200):
      return res.json()
    else:
      print(f"{id}")
      print(f"audio request failed with code {res.status_code} due to {res.reason}")

  def get_recco_artist_details(id):
    res = requests.get(f"{base_recco}/artist/{id}", headers=headers)
    return res.json()


In [ ]:
BATCH_SIZE = 40
all_results = []
for i in range(0, len(song_ids), BATCH_SIZE):
    batch = song_ids[i:i + BATCH_SIZE]
    results=get_recco_song_details(batch)['content']
    for r in results:
        song_id = r['id']
        all_results.append(r)
    time.sleep(1)

song__details=all_results

for item in all_results:
    res = get_recco_audio_analysis(item['id'])
    item.update(res)


In [ ]:
audio_features = pd.DataFrame(all_results)
dropped = ['ean', 'id', 'availableCountries', 'isrc', 'upc', 'href']
audio_features.drop(labels=dropped, inplace=True,errors='ignore')


df_artists = pd.DataFrame(artists)


,name,id,url,mbid,tag
0,Taylor Swift,06HL4z0CvFAxyc27GXpf02,https://open.spotify.com/artist/06HL4z0CvFAxyc...,20244d07-534f-4eff-b4d4-930878889970,pop
1,HUNTR/X,2yNNYQBChuox9A5Ka93BIn,https://open.spotify.com/artist/2yNNYQBChuox9A...,dc7fd424-6ba4-45cc-b407-7716c7bb3605,k-pop
2,EJAE,0RMJOzHDhAKY1o2j0W0vxY,https://open.spotify.com/artist/0RMJOzHDhAKY1o...,0f0a0c02-0aa6-4fc0-aad0-bb2e9d5bcbbb,NaN
3,AUDREY NUNA,0Wwji82sLA0Hcvtuak3omb,https://open.spotify.com/artist/0Wwji82sLA0Hcv...,7b54e5e4-82f7-4dfe-9cd8-fc8d9f6815e3,alternative r&b
4,REI AMI,6U1dV7aL68N7Gb0Naq34V5,https://open.spotify.com/artist/6U1dV7aL68N7Gb...,59adde12-dd9d-40c6-9906-bd4357633915,NaN
...,...,...,...,...,...
86,Tucker Wetmore,4sCKpwwEsgReZxjtKFm2A0,https://open.spotify.com/artist/4sCKpwwEsgReZx...,bac63106-16c3-4a31-80a4-d269ec27ab96,country
87,Yeat,3qiHUAX7zY4Qnjx8TNUzVx,https://open.spotify.com/artist/3qiHUAX7zY4Qnj...,9e6a6a2f-f696-4cc0-87f1-e4f712e46802,cloud rap
88,Post Malone,246dkjvS1zLTtiykXe5h60,https://open.spotify.com/artist/246dkjvS1zLTti...,b1e26560-60e5-4236-bbdb-9aa5a8d5ee19,hip hop
89,Loe Shimmy,6UIpxj5ggLdOebFVCOxVax,https://open.spotify.com/artist/6UIpxj5ggLdOeb...,ed411d15-00f4-4e56-b264-94e0d2f64ffe,NaN


In [ ]:
# import re
# import unicodedata
# new_songs = []
# sp = SpotifyAPI()

# def normalize_artist_name(artist) -> str:
#     #only need to get one artist if there are multiple
   
#     for c in ["Featuring", ",", "&", ":"]:
#         if c in artist:
#             artist = artist.split(c)[0]
#     return artist

# for entry in chart:
#     song = entry.title
#     artist = normalize_artist_name(entry.artist)
#     new_songs.append({
#         "song" : song,
#         "artist": artist
#     })
#     # new_songs.append(entry.title)
#     # print(entry)

# # sp.search_song(song="Take Me Thru Dere", artist="YKNIECE")
# new_songs
# # sp.search_artist("yg")

[{'song': 'The Fate Of Ophelia', 'artist': 'Taylor Swift'},
 {'song': 'Golden', 'artist': 'HUNTR/X'},
 {'song': 'Ordinary', 'artist': 'Alex Warren'},
 {'song': 'Man I Need', 'artist': 'Olivia Dean'},
 {'song': 'Opalite', 'artist': 'Taylor Swift'},
 {'song': 'Daisies', 'artist': 'Justin Bieber'},
 {'song': 'Mutt', 'artist': 'Leon Thomas'},
 {'song': 'Folded', 'artist': 'Kehlani'},
 {'song': 'I Got Better', 'artist': 'Morgan Wallen'},
 {'song': 'Back To Friends', 'artist': 'sombr'},
 {'song': 'All I Want For Christmas Is You', 'artist': 'Mariah Carey'},
 {'song': "Travelin' Soldier", 'artist': 'Cody Johnson '},
 {'song': 'Last Christmas', 'artist': 'Wham!'},
 {'song': 'What I Want', 'artist': 'Morgan Wallen '},
 {'song': 'Soda Pop', 'artist': 'Saja Boys'},
 {'song': 'It Depends', 'artist': 'Chris Brown '},
 {'song': 'Elizabeth Taylor', 'artist': 'Taylor Swift'},
 {'song': 'Tears', 'artist': 'Sabrina Carpenter'},
 {'song': "Rockin' Around The Christmas Tree", 'artist': 'Brenda Lee'},
 {'s

In [13]:
from extract import SpotifyAPI, MusicBrainzAPI, ReccoBeats, BillBoardChart
from datetime import date
import re
import unicodedata

sp = SpotifyAPI()
def get_spotify_song_ids_and_artists(chart) -> tuple [list, list]:
    num = 0 
    artists = []
    song_ids = []
    for song in chart:
        # print(num)
        artist = normalize_artist_name(song.artist)
        res = sp.search_song(song.title, artist)
        print(res)
        #if the song is found, add song and artist
        if res and res!= -1:
            song_ids.append(res['id'])
            existing_ids = {a["id"] for a in artists}
            for item in res.get("artists", []):
                artist_id = item.get("id")
                artist_url = item.get("external_urls", {}).get("spotify")
                artist_name = item.get("name")
                if artist_id and artist_id not in existing_ids:
                    artists.append({
                        "name": artist_name,
                        "id": artist_id,
                        "url": artist_url
                    })
                    existing_ids.add(artist_id)
        num+=1
    return (song_ids, artists)


def normalize_artist_name(artist) -> str:
    #only need to get one artist if there are multiple
    if artist == "JEONGYEON, JIHYO & CHAEYOUNG Of TWICE":
        artist = "TWICE"
    #special case
    else:
        for c in ["Featuring", ",", "&", ":"]:
            if c in artist:
                artist = artist.split(c)[0]
    
    return artist
    




today = date.today()
chart = BillBoardChart(date=today).data
song, artist = get_spotify_song_ids_and_artists(chart)
    
# print(song)
# print(artist)


{'album': {'album_type': 'album', 'artists': [{'external_urls': {'spotify': 'https://open.spotify.com/artist/06HL4z0CvFAxyc27GXpf02'}, 'href': 'https://api.spotify.com/v1/artists/06HL4z0CvFAxyc27GXpf02', 'id': '06HL4z0CvFAxyc27GXpf02', 'name': 'Taylor Swift', 'type': 'artist', 'uri': 'spotify:artist:06HL4z0CvFAxyc27GXpf02'}], 'available_markets': ['AR', 'AU', 'AT', 'BE', 'BO', 'BR', 'BG', 'CA', 'CL', 'CO', 'CR', 'CY', 'CZ', 'DK', 'DO', 'DE', 'EC', 'EE', 'SV', 'FI', 'FR', 'GR', 'GT', 'HN', 'HK', 'HU', 'IS', 'IE', 'IT', 'LV', 'LT', 'LU', 'MY', 'MT', 'MX', 'NL', 'NZ', 'NI', 'NO', 'PA', 'PY', 'PE', 'PH', 'PL', 'PT', 'SG', 'SK', 'ES', 'SE', 'CH', 'TW', 'TR', 'UY', 'US', 'GB', 'AD', 'LI', 'MC', 'ID', 'JP', 'TH', 'VN', 'RO', 'IL', 'ZA', 'SA', 'AE', 'BH', 'QA', 'OM', 'KW', 'EG', 'MA', 'DZ', 'TN', 'LB', 'JO', 'PS', 'IN', 'KZ', 'MD', 'UA', 'AL', 'BA', 'HR', 'ME', 'MK', 'RS', 'SI', 'KR', 'BD', 'PK', 'LK', 'GH', 'KE', 'NG', 'TZ', 'UG', 'AG', 'AM', 'BS', 'BB', 'BZ', 'BT', 'BW', 'BF', 'CV', 'CW', 'D

In [20]:
sp = SpotifyAPI()
sp.get_song("2RkZ5LkEzeHGRsmDqKwmaJ")

https://api.spotify.com/v1/tracks/2RkZ5LkEzeHGRsmDqKwmaJ


{'album': {'album_type': 'album',
  'artists': [{'external_urls': {'spotify': 'https://open.spotify.com/artist/0fTSzq9jAh4c36UVb4V7CB'},
    'href': 'https://api.spotify.com/v1/artists/0fTSzq9jAh4c36UVb4V7CB',
    'id': '0fTSzq9jAh4c36UVb4V7CB',
    'name': 'Alex Warren',
    'type': 'artist',
    'uri': 'spotify:artist:0fTSzq9jAh4c36UVb4V7CB'}],
  'available_markets': ['AR',
   'AU',
   'AT',
   'BE',
   'BO',
   'BR',
   'BG',
   'CA',
   'CL',
   'CO',
   'CR',
   'CY',
   'CZ',
   'DK',
   'DO',
   'DE',
   'EC',
   'EE',
   'SV',
   'FI',
   'FR',
   'GR',
   'GT',
   'HN',
   'HK',
   'HU',
   'IS',
   'IE',
   'IT',
   'LV',
   'LT',
   'LU',
   'MY',
   'MT',
   'MX',
   'NL',
   'NZ',
   'NI',
   'NO',
   'PA',
   'PY',
   'PE',
   'PH',
   'PL',
   'PT',
   'SG',
   'SK',
   'ES',
   'SE',
   'CH',
   'TW',
   'TR',
   'UY',
   'US',
   'GB',
   'AD',
   'LI',
   'MC',
   'ID',
   'JP',
   'TH',
   'VN',
   'RO',
   'IL',
   'ZA',
   'SA',
   'AE',
   'BH',
   'QA',
   'OM',


In [10]:
##testing
from extract import SpotifyAPI, MusicBrainzAPI, ReccoBeats, BillBoardChart
from datetime import date
import re
import unicodedata
import time
import pandas as pd
import billboard


def get_chart_dataframe(chart):
    rows = []
    print(chart)
    for entry in chart.entries:
        print("hello")
        rows.append({
            "title": entry.title,
            "artist": entry.artist,
            "rank": entry.rank,
            "isNew": entry.isNew,
            "weeks": entry.weeks,
            "peakPos": entry.peakPos
        })

    return pd.DataFrame(rows)

today = date.today()
print(today)
# chart_test = billboard.ChartData('hot-100')
chart_test = BillBoardChart('2025-11-23').data
# print(chart_test.entries)

df_chart_test = get_chart_dataframe(chart_test)
# print(df_chart_test)


2025-11-24
hot-100 chart from 2025-11-23
-----------------------------
